In [6]:
from pathlib import Path 
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import auc, confusion_matrix, roc_curve
import matplotlib.pyplot as plt
import seaborn as sns


# Preparação da Pipeline e Função de Avaliação Modular

In [2]:
# 1. Resolução do caminho dos dados (ajustado para a pasta 'notebooks/')
data_processed = Path("../data/processed")
if not data_processed.exists():
    data_processed = Path("data/processed")

caminho_train = data_processed / "features_train.csv"
caminho_test = data_processed / "features_test.csv"

# 2. Carregamento dos arquivos CSV
df_train = pd.read_csv(caminho_train)
df_test = pd.read_csv(caminho_test)

# 3. Separação entre Metadados e Matrizes de Características (X)
cols_meta = ["caminho", "classe", "split"]
cols_features = [col for col in df_train.columns if col not in cols_meta]

# Matrizes numéricas brutas de entrada para os modelos
X_train_raw = df_train[cols_features].values
X_test_raw = df_test[cols_features].values

# Metadados e Rótulos originais
y_train_labels = df_train["classe"].values
y_test_labels = df_test["classe"].values

# 4. Criação do Rótulo Binário para Detecção de Anomalias:
# Convenção padrão do Scikit-Learn (OneClassSVM, IsolationForest, LOF):
# +1 = Peça Normal ('good')
# -1 = Anomalia / Peça Defeituosa (qualquer categoria diferente de 'good')
y_test_binary = np.where(y_test_labels == "good", 1, -1)

# 5. Exibição do resumo de carregamento
print("==================================================")
print(" RESUMO DO CARREGAMENTO DOS DATASETS ")
print("==================================================")
print(
    f"✓ Treino (X_train_raw): {X_train_raw.shape[0]} amostras,"
    f" {X_train_raw.shape[1]} características"
)
print(
    f"✓ Teste (X_test_raw) : {X_test_raw.shape[0]} amostras,"
    f" {X_test_raw.shape[1]} características"
)
print("\nDistribuição de categorias no conjunto de Teste:")
print(pd.Series(y_test_labels).value_counts())
print("\nDistribuição dos rótulos binários (+1 = Normal, -1 = Anomalia):")
print(pd.Series(y_test_binary).value_counts())

 RESUMO DO CARREGAMENTO DOS DATASETS 
✓ Treino (X_train_raw): 320 amostras, 25413 características
✓ Teste (X_test_raw) : 160 amostras, 25413 características

Distribuição de categorias no conjunto de Teste:
good                 41
scratch_neck         25
manipulated_front    24
scratch_head         24
thread_side          23
thread_top           23
Name: count, dtype: int64

Distribuição dos rótulos binários (+1 = Normal, -1 = Anomalia):
-1    119
 1     41
Name: count, dtype: int64


In [5]:
# 1. Instancia o escalonador
scaler = StandardScaler()

# 2. Ajusta o fit EXCLUSIVAMENTE com os dados de treino ('good')
scaler.fit(X_train_raw)

# 3. Transforma as matrizes de treino e teste
X_train = scaler.transform(X_train_raw)
X_test = scaler.transform(X_test_raw)

# 4. Verificação da padronização
print("==================================================")
print(" PADRONIZAÇÃO COM STANDARDSCALER ")
print("==================================================")
print(f"✓ Formato X_train escalonado: {X_train.shape}")
print(f"✓ Formato X_test escalonado : {X_test.shape}")
print(
    f"\nMédia aproximada das colunas no treino (deve ser ~0):"
    f" {np.mean(X_train):.4f}"
)
print(
    f"Desvio padrão médio das colunas no treino (deve ser ~1):"
    f" {np.mean(np.std(X_train, axis=0)):.4f}"
)

 PADRONIZAÇÃO COM STANDARDSCALER 
✓ Formato X_train escalonado: (320, 25413)
✓ Formato X_test escalonado : (160, 25413)

Média aproximada das colunas no treino (deve ser ~0): 0.0000
Desvio padrão médio das colunas no treino (deve ser ~1): 1.0000


In [7]:
def avaliar_detector_anomalias(
    modelo,
    X_test: np.ndarray,
    y_test_binary: np.ndarray,
    y_test_labels: np.ndarray,
    nome_modelo: str = "Detector de Anomalias",
):
    """Executa a avaliação completa de um modelo de detecção de anomalias:
    - Predição dos rótulos binários (+1 = Normal, -1 = Anomalia)
    - Cálculo de pontuação de decisão (decision_function)
    - Matriz de Confusão Binária
    - Curva ROC e AUC
    - Sensibilidade (Recall) detalhada por tipo de defeito
    """
    # 1. Predição e pontuação de decisão
    y_pred = modelo.predict(X_test)  # +1 (Normal) ou -1 (Anomalia)

    if hasattr(modelo, "decision_function"):
        scores = modelo.decision_function(X_test)
    else:
        scores = modelo.score_samples(X_test)

    # 2. Matriz de Confusão Binária (Mapeamento: Normal=1, Anomalia=-1)
    # Rótulos: [1, -1] -> Matriz: [[TP, FN], [FP, TN]] em relação à classe Normal
    cm = confusion_matrix(y_test_binary, y_pred, labels=[1, -1])

    # 3. Métricas Globais
    tn, fp, fn, tp = confusion_matrix(
        y_test_binary == -1, y_pred == -1
    ).ravel()

    acuracia = (tp + tn) / (tp + tn + fp + fn)
    sensibilidade_geral = (
        tp / (tp + fn) if (tp + fn) > 0 else 0
    )  # Defeitos capturados
    especificidade = (
        tn / (tn + fp) if (tn + fp) > 0 else 0
    )  # Normais aprovados corretamente

    # 4. Curva ROC (Invertemos os scores pois pontuações menores indicam anomalias)
    fpr, tpr, _ = roc_curve(
        y_test_binary == -1, -scores
    )  # Classe positiva = Anomalia (-1)
    roc_auc = auc(fpr, tpr)

    # 5. Impressão do Relatório de Desempenho
    print("=" * 60)
    print(f" RELATÓRIO DE AVALIAÇÃO: {nome_modelo.upper()}")
    print("=" * 60)
    print(f"✓ Acurácia Geral : {acuracia * 100:.2f}%")
    print(
        f"✓ Sensibilidade (Recall) : {sensibilidade_geral * 100:.2f}% (Defeitos"
        " detectados)"
    )
    print(
        f"✓ Especificidade : {especificidade * 100:.2f}% (Boms"
        " aprovados)"
    )
    print(f"✓ Falsos Negativos (Risco): {fn} (Defeitos aprovados como boms)")
    print(f"✓ Área Sob a Curva (AUC) : {roc_auc:.4f}\n")

    # 6. Sensibilidade por Categoria de Defeito
    df_analise = pd.DataFrame(
        {"classe_real": y_test_labels, "predicao_binaria": y_pred}
    )

    print("--------------------------------------------------")
    print(" SENSIBILIDADE POR CATEGORIA DE DEFEITO")
    print("--------------------------------------------------")

    categorias = np.unique(y_test_labels)

    for cat in categorias:
        sub = df_analise[df_analise["classe_real"] == cat]
        if cat == "good":
            taxa_acerto = (sub["predicao_binaria"] == 1).mean()
            print(
                f" - {cat:<18}: {taxa_acerto * 100:.2f}% aprovados corretamente"
                f" ({len(sub)} amostras)"
            )
        else:
            taxa_acerto = (sub["predicao_binaria"] == -1).mean()
            print(
                f" - {cat:<18}: {taxa_acerto * 100:.2f}% detectados como anomalia"
                f" ({len(sub)} amostras)"
            )

    print("--------------------------------------------------\n")

    # 7. Visualização Gráfica (Matriz de Confusão e Curva ROC)
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))

    # Plot 1: Matriz de Confusão
    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        cbar=False,
        xticklabels=["Normal (+1)", "Anomalia (-1)"],
        yticklabels=["Normal (+1)", "Anomalia (-1)"],
        ax=axes[0],
    )
    axes[0].set_title(f"Matriz de Confusão - {nome_modelo}")
    axes[0].set_xlabel("Predição do Modelo")
    axes[0].set_ylabel("Rótulo Real")

    # Plot 2: Curva ROC
    axes[1].plot(
        fpr,
        tpr,
        color="darkorange",
        lw=2,
        label=f"AUC = {roc_auc:.4f}",
    )
    axes[1].plot([0, 1], [0, 1], color="navy", lw=1.5, linestyle="--")
    axes[1].set_xlim([0.0, 1.0])
    axes[1].set_ylim([0.0, 1.05])
    axes[1].set_xlabel("Taxa de Falsos Positivos (FPR)")
    axes[1].set_ylabel("Taxa de Verdadeiros Positivos (TPR - Recall)")
    axes[1].set_title(f"Curva ROC - {nome_modelo}")
    axes[1].legend(loc="lower right")

    plt.tight_layout()
    plt.show()

    return {
        "acuracia": acuracia,
        "recall": sensibilidade_geral,
        "especificidade": especificidade,
        "auc": roc_auc,
        "falsos_negativos": fn,
    }